In [0]:
df = spark.table("aml_risk_prediction.gold.aml_ml_features").toPandas()

df.head()

In [0]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [0]:
df.isnull().sum()

In [0]:
df["laundering_flag"].value_counts()

df["laundering_flag"].value_counts(normalize=True) * 100

In [0]:
df.groupby("laundering_flag")[
    [
        "total_amount_paid",
        "total_amount_received",
        "transaction_count",
        "avg_amount_paid",
        "avg_transaction_value",
        "unique_payment_currencies"
    ]
].mean()

In [0]:
import matplotlib.pyplot as plt

# Convert column to float explicitly for matplotlib compatibility
df["total_amount_paid"] = df["total_amount_paid"].astype(float)

df.boxplot(column="total_amount_paid", by="laundering_flag")
plt.title("Transaction Amount vs Laundering")
plt.suptitle("")
plt.xlabel("Is Laundering")
plt.ylabel("Amount Paid")
plt.show()

In [0]:
features = [
        "total_amount_paid",
        "total_amount_received",
        "transaction_count",
        "avg_amount_paid",
        "avg_transaction_value",
        "unique_payment_currencies"
]

In [0]:
target = "laundering_flag"

In [0]:
X = df[features]
y = df[target]

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [0]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [0]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(
    X_train_scaled,
    y_train
)

In [0]:
y_pred = model.predict(X_test_scaled)

In [0]:
print(y_pred[:30])

In [0]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [0]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred
)

plt.title("AML Prediction Confusion Matrix")
plt.show()

In [0]:
risk_probability = model.predict_proba(
    X_test_scaled
)[:, 1]

In [0]:
results = df.loc[
    X_test.index,
    [
        "account_key",
        "total_amount_paid",
        "laundering_flag"
    ]
].copy()

results["risk_probability"] = risk_probability

def assign_risk(probability):
    if probability < 0.30:
        return "Low"
    elif probability < 0.70:
        return "Medium"
    else:
        return "High"

results["risk_level"] = results["risk_probability"].apply(assign_risk)

In [0]:
display(results.head(20))

In [0]:
spark_results = spark.createDataFrame(results)

spark_results.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("aml_risk_predictions")

In [0]:
results["risk_level"].value_counts()